In [5]:
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from czifile import CziFile
from cellpose import models
from skimage.filters import gaussian
from skimage import exposure
from skimage.measure import regionprops
from skimage.segmentation import find_boundaries
import os
import pandas as pd

OUT = "results/"
os.makedirs(OUT, exist_ok=True)
os.makedirs(f"{OUT}/images", exist_ok=True)
os.makedirs(f"{OUT}/per_file_csv", exist_ok=True)


# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------
FOLDER = "/Users/u5672638/Library/CloudStorage/OneDrive-UniversityofWarwick/FS_26/Hackathon/1.data/test/"   # <-- change
OVERLAP_THRESHOLD = 0.30

# Your original filtering thresholds
MIN_AREA = 5000
MIN_MEAN_INTENSITY = 7000

all_samples = []
all_ratios = []
all_representative_images = []


# Cellpose model (same as your script)
model = models.CellposeModel(gpu=False, model_type='nuclei')

# ---------------------------------------------------------
# LOOP OVER ALL FILES
# ---------------------------------------------------------
files = glob(FOLDER + "/*.czi")

for file in files:
    print("Processing:", file)

    # -----------------------------
    # Load + squeeze
    # -----------------------------
    czi = CziFile(file)
    img = np.squeeze(czi.asarray())

    # -----------------------------
    # Extract channels (your choice)
    # -----------------------------
    marker_img_orig = img[0].astype(np.float32)
    nuclei_img_orig = img[3].astype(np.float32)

    # -----------------------------
    # Smoothing + contrast (your exact pipeline)
    # -----------------------------
    marker_smooth = gaussian(marker_img_orig, sigma=2)
    marker_gamma = exposure.adjust_gamma(marker_smooth, gamma=0.5)

    nuclei_smooth = gaussian(nuclei_img_orig, sigma=2)
    nuclei_gamma = exposure.adjust_gamma(nuclei_smooth, gamma=0.7)

    marker_img = marker_gamma
    nuclei_img = nuclei_gamma

    # -----------------------------
    # Nuclei segmentation (your parameters)
    # -----------------------------
    nuc_masks, _, _ = model.eval(
        nuclei_img,
        channels=[0, 0],
        diameter=100,
        cellprob_threshold=0.1,
        flow_threshold=0.4
    )

    # -----------------------------
    # Marker segmentation (your parameters)
    # -----------------------------
    marker_masks, _, _ = model.eval(
        marker_img,
        channels=[0, 0],
        diameter=200,
        cellprob_threshold=1,
        flow_threshold=0.7
    )
        # -----------------------------
    # FILTER MARKER MASKS
    # -----------------------------
    filtered_marker_masks = np.zeros_like(marker_masks)

    for region in regionprops(marker_masks, intensity_image=marker_img_orig):
        if region.area >= MIN_AREA and region.mean_intensity >= MIN_MEAN_INTENSITY:
            filtered_marker_masks[region.coords[:,0], region.coords[:,1]] = region.label

    marker_masks = filtered_marker_masks

    # -----------------------------
    # Select nuclei overlapping marker mask
    # -----------------------------
    selected = set()
    for region in regionprops(nuc_masks):
        coords = region.coords
        overlap = np.sum(marker_masks[coords[:,0], coords[:,1]] > 0)
        frac = overlap / len(coords)
        if frac >= OVERLAP_THRESHOLD:
            selected.add(region.label)

    filtered_nuclei = np.zeros_like(nuc_masks)
    for region in regionprops(nuc_masks):
        if region.label in selected:
            filtered_nuclei[region.coords[:,0], region.coords[:,1]] = region.label

    # -----------------------------
    # Compute inside/outside ratio
    # -----------------------------
    ratios_this_file = []

    for region in regionprops(marker_masks, intensity_image=marker_img_orig):
        coords = region.coords
        intens = marker_img_orig[coords[:,0], coords[:,1]]
        inside = filtered_nuclei[coords[:,0], coords[:,1]] > 0

        intens_in = intens[inside]
        intens_out = intens[~inside]

        if len(intens_in) == 0 or len(intens_out) == 0:
            continue


        ratio = intens_in.mean() / intens_out.mean()
        ratios_this_file.append(ratio)
        basename = os.path.basename(file).replace(".czi", "")
        df_file = pd.DataFrame({"ratio_inside_outside": ratios_this_file})
        df_file.to_csv(f"{OUT}/per_file_csv/{basename}_ratios.csv", index=False)

    all_ratios.extend(ratios_this_file)
    all_samples.extend([basename] * len(ratios_this_file))

    # -----------------------------
    # Representative images (presentation-ready)
    # -----------------------------
    fig, ax = plt.subplots(1, 2, figsize=(12,6))

    ax[0].imshow(nuclei_img_orig, cmap='gray')
    ax[0].set_title("Original nuclei channel")
    ax[0].axis('off')

    # ax[1].imshow(marker_img_orig, cmap='gray')
    # ax[1].set_title("Original marker channel")
    # ax[1].axis('off')

    ax[1].imshow(marker_img_orig, cmap='gray')
    ax[1].set_title("Selected nuclei (cyan) + marker (yellow)")
    ax[1].axis('off')

    # outlines
    nuc_boundary = find_boundaries(filtered_nuclei > 0)
    marker_boundary = find_boundaries(marker_masks > 0)

    all_representative_images.append({
        "basename": basename,
        "nuclei": nuclei_img_orig,
        "marker": marker_img_orig,
        "nuc_boundary": nuc_boundary,
        "marker_boundary": marker_boundary
    })


    ax[0].contour(nuc_boundary, colors='cyan', linewidths=1)
    ax[1].contour(nuc_boundary, colors='cyan', linewidths=1)
    ax[1].contour(marker_boundary, colors='yellow', linewidths=1)

    basename = os.path.basename(file).replace(".czi", "")

    plt.tight_layout()
    plt.savefig(f"{OUT}/images/{basename}_representative.png",
                dpi=300, bbox_inches='tight')
    plt.close()


# ---------------------------------------------------------
# FINAL BOXPLOT
# ---------------------------------------------------------

plt.figure(figsize=(6,6))

# --- Boxplot ---
plt.boxplot(all_ratios, positions=[1])

# --- Scatter (all dots) ---
x = np.ones(len(all_ratios))
x_jitter = x + (np.random.rand(len(all_ratios)) - 0.5) * 0.1
plt.scatter(x_jitter, all_ratios, color='black', alpha=0.6, s=20)

plt.ylabel("Mean intensity ratio (inside / outside)")
plt.title("Marker enrichment across all files")
plt.xticks([1], ["All ratios"])

# --- SAVE ---
plt.savefig(f"{OUT}/boxplot_all_ratios.png", dpi=300, bbox_inches='tight')
plt.close()



df_all = pd.DataFrame({
    "ratio_inside_outside": all_ratios,
    "sample": all_samples   # we will create this list
})
df_all.to_csv(f"{OUT}/all_ratios.csv", index=False)


import math

n = len(all_representative_images)
cols = 4
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(10, 5 * rows))
axes = axes.ravel()

for i, item in enumerate(all_representative_images):
    ax = axes[i]
    ax.imshow(item["marker"], cmap='gray')
    ax.set_title(item["basename"])
    ax.axis('off')

    # overlay boundaries
    ax.contour(item["nuc_boundary"], colors='cyan', linewidths=1)
    ax.contour(item["marker_boundary"], colors='yellow', linewidths=1)

# hide unused axes
for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig(f"{OUT}/images/all_samples_overview.png", dpi=300, bbox_inches='tight')
plt.close()


mean_this = np.mean(ratios_this_file)
sd_this = np.std(ratios_this_file)
print(f"{basename}: mean={mean_this:.3f}, sd={sd_this:.3f}")

Processing: /Users/u5672638/Library/CloudStorage/OneDrive-UniversityofWarwick/FS_26/Hackathon/1.data/test/6_6.czi
Processing: /Users/u5672638/Library/CloudStorage/OneDrive-UniversityofWarwick/FS_26/Hackathon/1.data/test/6_5.czi
Processing: /Users/u5672638/Library/CloudStorage/OneDrive-UniversityofWarwick/FS_26/Hackathon/1.data/test/6_4.czi
Processing: /Users/u5672638/Library/CloudStorage/OneDrive-UniversityofWarwick/FS_26/Hackathon/1.data/test/6_1.czi
Processing: /Users/u5672638/Library/CloudStorage/OneDrive-UniversityofWarwick/FS_26/Hackathon/1.data/test/6_3.czi
Processing: /Users/u5672638/Library/CloudStorage/OneDrive-UniversityofWarwick/FS_26/Hackathon/1.data/test/6_2.czi
Processing: /Users/u5672638/Library/CloudStorage/OneDrive-UniversityofWarwick/FS_26/Hackathon/1.data/test/7_5.czi
Processing: /Users/u5672638/Library/CloudStorage/OneDrive-UniversityofWarwick/FS_26/Hackathon/1.data/test/7_4.czi
Processing: /Users/u5672638/Library/CloudStorage/OneDrive-UniversityofWarwick/FS_26/Hack